In [1]:
# ==============================================================================
# CELL 0: PIENZA CLOUD BOOTSTRAP (DUAL-LAKE ARCHITECTURE)
# ==============================================================================
# Environment: Public Google Colab (External Forge)
# Purpose: Authentication and Dual BigQuery connection (pienza_mini & pienza_big)
# ==============================================================================

# --- 1. CLOUD PASSPORT ---
from google.colab import auth
print("🔐 Requesting Google Cloud Passport...")
auth.authenticate_user()
print("✅ Authentication Successful.")

# --- 2. CORE IMPORTS ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
import warnings
warnings.filterwarnings('ignore')

# --- 3. SOVEREIGN COORDINATES ---
PROJECT_ID    = '645009831643'
DATASET_MINI  = 'pienza_mini'
DATASET_BIG   = 'pienza_big'

try:
    client = bigquery.Client(project=PROJECT_ID)
    print(f"✅ BigQuery Client Active: {PROJECT_ID}")

    # Probing both datasets
    client.query(f"SELECT 1 FROM `{PROJECT_ID}.{DATASET_MINI}.INFORMATION_SCHEMA.TABLES` LIMIT 1").result()
    client.query(f"SELECT 1 FROM `{PROJECT_ID}.{DATASET_BIG}.INFORMATION_SCHEMA.TABLES` LIMIT 1").result()
    print("✅ Sovereign Data Connection: STABLE across both datasets.")
except Exception as e:
    print(f"🔴 CRITICAL: Cloud connection failed. Details: {e}")

# --- 4. VISUAL CANON ---
PIENZA_PURPLE = '#440154'
PIENZA_TEAL   = '#21918c'
PIENZA_GREY   = '#FAFAFA'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': PIENZA_GREY,
    'axes.facecolor': PIENZA_GREY,
    'axes.titlecolor': PIENZA_PURPLE,
    'figure.titlesize': 24,
    'figure.titleweight': 'bold'
})

print("\n--- PIENZA DUAL-LAKE SYSTEM READY ---")

🔐 Requesting Google Cloud Passport...
✅ Authentication Successful.
✅ BigQuery Client Active: 645009831643
✅ Sovereign Data Connection: STABLE across both datasets.

--- PIENZA DUAL-LAKE SYSTEM READY ---


In [2]:
# Quick Schema Probe
query = f"""
    SELECT column_name, data_type
    FROM `{PROJECT_ID}.{DATASET_BIG}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = 'synthetic_manifold_v8_enriched'
"""
df_schema_big = client.query(query).to_dataframe()
print(df_schema_big['column_name'].tolist())

['upfront_fare', 'est_trip_time_sec', 'est_trip_dist_km', 'time_to_pickup_sec', 'dist_to_pickup_km', 'hour_of_day', 'day_of_week', 'product_category_fk', 'dropoff_zone_id', 'pickup_zone_id', 'reason_primary_fk', 'product_name', 'dropoff_name', 'pickup_name']


In [3]:
# ==============================================================================
# CELL 1: THE BLUEPRINT (FEATURE & TARGET MANIFEST)
# ==============================================================================
# Purpose: Define the exact vector space that both Reality and Simulation must respect.
# ==============================================================================

print("📐 Definiendo el Blueprint Dimensional...")

# 1. Los Features Físicos y Semánticos
CORE_FEATURES = [
    'upfront_fare',
    'est_trip_time_sec',
    'est_trip_dist_km',
    'time_to_pickup_sec',
    'dist_to_pickup_km',
    'hour_of_day',        # Tiempo
    'day_of_week',        # Tiempo
    'product_category_fk','product_name',   # Producto (ID + Text)
    'dropoff_zone_id',    'dropoff_name',   # Destino (ID + Text)
    'pickup_zone_id',     'pickup_name'     # Origen (ID + Text)
]

# 2. La Variable Dependiente (El comportamiento del Agente)
TARGET = 'reason_primary_fk'

# 3. Lista final para los queries SQL
QUERY_COLUMNS = CORE_FEATURES + [TARGET]

# Convertimos a string para inyectarlo en SQL
sql_select_clause = ", ".join([f"`{col}`" for col in QUERY_COLUMNS])

print(f"✅ Blueprint fijado: {len(CORE_FEATURES)} Features + 1 Target ({TARGET})")

📐 Definiendo el Blueprint Dimensional...
✅ Blueprint fijado: 13 Features + 1 Target (reason_primary_fk)


In [5]:
# ==============================================================================
# CELL 2: THE SOVEREIGN EXTRACTION & THE SCHISM (TSTR SPLIT)
# ==============================================================================
from sklearn.model_selection import train_test_split

print("🚀 Iniciando Extracción Dual-Lake (Arquitectura Limpia)...")

# --- 1. INGESTA SINTÉTICA (df_synth) ---
TABLE_SYNTH = "synthetic_manifold_v8_enriched"

query_synth = f"""
    SELECT {sql_select_clause}
    FROM `{PROJECT_ID}.{DATASET_BIG}.{TABLE_SYNTH}`
"""
print(f"   📥 Descargando Manifold Sintético ({TABLE_SYNTH})...")
df_synth = client.query(query_synth).to_dataframe()
print(f"   ✅ df_synth en memoria: {df_synth.shape[0]:,} filas.")

# --- 2. INGESTA REAL (df_real_full) ---
TABLE_REAL = "v_ML_Supervised"

query_real = f"""
    SELECT {sql_select_clause}
    FROM `{PROJECT_ID}.{DATASET_MINI}.{TABLE_REAL}`
    WHERE {TARGET} IS NOT NULL
"""
print(f"   📥 Descargando Realidad ({TABLE_REAL})...")
try:
    df_real_full = client.query(query_real).to_dataframe()
    print(f"   ✅ Realidad en memoria: {df_real_full.shape[0]:,} filas.")
except Exception as e:
    print(f"❌ ALERTA EN INGESTA REAL: {e}")
    print("💡 Tip forense: Si BigQuery se queja de 'product_name' o 'pickup_name', es porque en la vista real tienen otro nombre (ej. product_category_fk). ¡Lo ajustamos con un simple ALIAS si chilla!")

# --- 3. EL CISMA (Train/Test Split de la Realidad) ---
if 'df_real_full' in locals():
    print("\n⚔️ Ejecutando El Cisma (80% Train / 20% Holdout)...")

    df_train_real, df_holdout = train_test_split(
        df_real_full,
        test_size=0.20,
        random_state=42,
        stratify=df_real_full[TARGET]
    )

    print(f"   🧠 df_train_real (Baseline): {df_train_real.shape[0]:,} filas.")
    print(f"   🛡️ df_holdout (Test Intocable): {df_holdout.shape[0]:,} filas.")
    print("\n🏁 INFRAESTRUCTURA DE DATOS TSTR COMPLETADA.")

🚀 Iniciando Extracción Dual-Lake (Arquitectura Limpia)...
   📥 Descargando Manifold Sintético (synthetic_manifold_v8_enriched)...
   ✅ df_synth en memoria: 1,010,001 filas.
   📥 Descargando Realidad (v_ML_Supervised)...
❌ ALERTA EN INGESTA REAL: 400 Unrecognized name: product_name at [2:165]; reason: invalidQuery, location: query, message: Unrecognized name: product_name at [2:165]

Location: US
Job ID: d4d43605-8d19-4479-aa0b-86a4cbbc2a51

💡 Tip forense: Si BigQuery se queja de 'product_name' o 'pickup_name', es porque en la vista real tienen otro nombre (ej. product_category_fk). ¡Lo ajustamos con un simple ALIAS si chilla!
